In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# =========================================================================
# 🚢 CHARGEMENT INITIAL DES DONNÉES & DIAGNOSTIC
# =========================================================================
print("=========================================================")
print("🚢 DEBUT DE L'ANALYSE DU JEU DE DONNEES TITANIC")
print("=========================================================\n")

# URL officielle du fichier train.csv du Titanic
url_titanic = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(url_titanic)

print("--- 1. Aperçu des 5 premières lignes ---")
print(df_titanic.head(5))

print("\n--- 2. Informations globales sur les colonnes ---")
print(df_titanic.info())

print("\n--- 3. Détection initiale des valeurs manquantes (NaN) ---")
print(df_titanic.isnull().sum())


# =========================================================================
# 🧹 EXERCICE 1 : DÉTECTION ET SUPPRESSION DES DOUBLONS
# =========================================================================
print("\n=========================================================")
print("🧹 EXERCICE 1 : SUPPRESSION DES DOUBLONS")
print("=========================================================\n")

lignes_avant = df_titanic.shape[0]
nb_doublons = df_titanic.duplicated().sum()
print(f"🔍 Nombre de lignes en double détectées : {nb_doublons}")

# Suppression des doublons s'il y en a
df_titanic.drop_duplicates(inplace=True)
print(f"✅ Vérification : {df_titanic.shape[0]} lignes restantes après nettoyage des doublons.")


# =========================================================================
# 🛠️ EXERCICE 2 : GESTION DES VALEURS MANQUANTES
# =========================================================================
print("\n=========================================================")
print("🛠️ EXERCICE 2 : GESTION DES VALEURS MANQUANTES")
print("=========================================================\n")

# Stratégie A : Remplacement des âges manquants par la médiane (Pandas)
mediane_age = df_titanic['Age'].median()
df_titanic['Age'] = df_titanic['Age'].fillna(mediane_age)
print(f"✅ Remplacement des âges manquants par la médiane : {mediane_age} ans.")

# Stratégie B : Remplissage de 'Cabin' par une constante 'Inconnu' (Pandas - Plus stable que SimpleImputer ici)
df_titanic['Cabin'] = df_titanic['Cabin'].fillna('Inconnu')
print("✅ Remplacement des valeurs manquantes de 'Cabin' par la constante 'Inconnu'.")

# Stratégie C : Suppression des lignes pour lesquelles 'Embarked' est manquant (Pandas)
df_titanic.dropna(subset=['Embarked'], inplace=True)
print("✅ Suppression des lignes ayant un port d'embarquement ('Embarked') manquant.")

print("\n--- Vérification finale des données manquantes ---")
print(df_titanic.isnull().sum())


# =========================================================================
# ✨ EXERCICE 3 : INGÉNIERIE DES FONCTIONNALITÉS (FEATURE ENGINEERING)
# =========================================================================
print("\n=========================================================")
print("✨ EXERCICE 3 : INGENIERIE DES FONCTIONNALITÉS")
print("=========================================================\n")

# 1. Création de la taille de la famille (FamilySize)
df_titanic['FamilySize'] = df_titanic['SibSp'] + df_titanic['Parch'] + 1

# 2. Extraction du Titre depuis la colonne 'Name' (Correction du SyntaxWarning avec r'...')
df_titanic['Title'] = df_titanic['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Regroupement des titres rares pour stabiliser les futures analyses
titres_rares = ['Dr', 'Rev', 'Col', 'Major', 'Mlle', 'Countess', 'Ms', 'Lady', 'Jonkheer', 'Don', 'Capt', 'Sir', 'Mme']
df_titanic['Title'] = df_titanic['Title'].replace(titres_rares, 'Rare')

print("Aperçu des nouvelles colonnes créées :")
print(df_titanic[['Name', 'FamilySize', 'Title']].head(5))


# =========================================================================
# 📊 EXERCICE 4 : DÉTECTION ET TRAITEMENT DES VALEURS ABERRANTES
# =========================================================================
print("\n=========================================================")
print("📊 EXERCICE 4 : VALEURS ABERRANTES (PRIX DU BILLET)")
print("=========================================================\n")

# Calcul du seuil au quantile 98% pour la variable 'Fare'
seuil_98 = df_titanic['Fare'].quantile(0.98)
print(f"Le seuil déterminé pour le quantile 98% est de : {seuil_98:.2f}$")

fare_max_avant = df_titanic['Fare'].max()

# Plafonnement (Capping) des valeurs supérieures au quantile 98%
df_titanic['Fare'] = np.where(df_titanic['Fare'] > seuil_98, seuil_98, df_titanic['Fare'])
fare_max_apres = df_titanic['Fare'].max()

print(f"💰 Prix maximum AVANT plafonnement : {fare_max_avant:.2f}$")
print(f"💰 Prix maximum APRÈS plafonnement : {fare_max_apres:.2f}$")
print("✅ Valeurs aberrantes traitées par plafonnement.")


# =========================================================================
# ⚖️ EXERCICE 5 : STANDARDISATION ET NORMALISATION DES DONNÉES
# =========================================================================
print("\n=========================================================")
print("⚖️ EXERCICE 5 : MISE À L'ÉCHELLE DES VARIABLES NUMÉRIQUES")
print("=========================================================\n")

scaler_standard = StandardScaler()
scaler_minmax = MinMaxScaler()

# StandardScaler pour l'Age (Distribution normale)
df_titanic['Age_scaled'] = scaler_standard.fit_transform(df_titanic[['Age']])

# MinMaxScaler pour le Fare (Distribution asymétrique/bornée)
df_titanic['Fare_scaled'] = scaler_minmax.fit_transform(df_titanic[['Fare']])

print("Aperçu des variables d'origine face aux variables normalisées :")
print(df_titanic[['Age', 'Age_scaled', 'Fare', 'Fare_scaled']].head(5))


# =========================================================================
# 🏷️ EXERCICE 6 : ENCODAGE DES CARACTÉRISTIQUES CATÉGORIELLES
# =========================================================================
print("\n=========================================================")
print("🏷️ EXERCICE 6 : ENCODAGE DES VARIABLES CATEGORIELLES")
print("=========================================================\n")

# Sélection des variables nominales à transformer en One-Hot (0 ou 1)
colonnes_categoriques = ['Sex', 'Embarked', 'Title']

# Encodage One-Hot (drop_first=True évite le piège de la multi-colinéarité)
df_encoded = pd.get_dummies(df_titanic, columns=colonnes_categoriques, drop_first=True)

print(f"Nombre total de colonnes après encodage One-Hot : {df_encoded.shape[1]}")


# =========================================================================
# ⏳ EXERCICE 7 : TRANSFORMATION ET ENCODAGE DES TRANCHES D'ÂGE
# =========================================================================
print("\n=========================================================")
print("⏳ EXERCICE 7 : DISCRÉTISATION DE L'ÂGE ET ENCODAGE FINAL")
print("=========================================================\n")

# 1. Définition des tranches de vie (bins) et des étiquettes associées (labels)
bacs = [0, 12, 18, 60, 100]
noms_categories = ['Enfant', 'Adolescent', 'Adulte', 'Senior']

# 2. Découpage (Discrétisation)
df_encoded['AgeGroup'] = pd.cut(df_encoded['Age'], bins=bacs, labels=noms_categories)

# 3. Encodage One-Hot final des groupes d'âge créés
df_final = pd.get_dummies(df_encoded, columns=['AgeGroup'], drop_first=False)

print("Aperçu final des colonnes de tranches d'âge encodées :")
colonnes_age_group = [col for col in df_final.columns if 'AgeGroup' in col]
print(df_final[colonnes_age_group].head(5))

print("\n=========================================================")
print("🎉 PIPELINE TERMINÉ AVEC SUCCÈS ! Votre jeu de données est prêt.")
print("=========================================================")

🚢 DEBUT DE L'ANALYSE DU JEU DE DONNEES TITANIC

--- 1. Aperçu des 5 premières lignes ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0           